# 04 — Sensitivity analysis

`03_backtest_results.ipynb` reports what one frozen configuration produced. This notebook
asks the only question that determines whether those numbers mean anything:

> **How much of the result is the strategy, and how much is the arbitrary choices?**

Five parameters were fixed before the out-of-sample period was scored: the z-score
**window**, the **entry** and **exit** bands, the **cost** assumption, the **split date**,
and the **hedge specification**. Each was picked on defensible a priori grounds. None was
picked because it worked. Every one is varied below, one at a time, with everything else —
including the in-sample-only hedge fit — held exactly as the study ran it.

**This is not a parameter search.** Nothing here feeds back into `configs/pairs.yaml`. If
these sweeps showed a better window, adopting it would convert an out-of-sample result into
an in-sample one, which is the precise failure the whole project is built to avoid. The
sweeps are reported as findings about *stability*, and stability is what turns out to be
missing.

Every sweep re-runs the full pipeline through `pairs_teardown.study.run_pair`, so these are
the same code paths `make run` uses — not a reimplementation.


In [ ]:
import dataclasses
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pairs_teardown.config import load_config
from pairs_teardown.data.loaders import load_or_download
from pairs_teardown.study import run_pair

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load_config(ROOT / "configs" / "pairs.yaml")
prices = load_or_download(
    list(cfg.tickers), cfg.data.start, cfg.data.end, ROOT / cfg.data.cache_dir
)
PAIRS = [p.name for p in cfg.pairs]


def with_signal(base, **kw):
    return dataclasses.replace(base, signal=dataclasses.replace(base.signal, **kw))


def oos(variant, field="total_return"):
    """Net out-of-sample metric for every pair under one config variant, in %."""
    return pd.Series(
        {p.name: run_pair(p, prices, variant).metrics["out_of_sample"]["net"][field] * 100
         for p in cfg.pairs}
    )


BASELINE = oos(cfg)
print("baseline (as configured): net OOS total return %")
print(f"  mean {BASELINE.mean():+.2f}   median {BASELINE.median():+.2f}   "
      f"sd {BASELINE.std():.1f}   positive {int((BASELINE > 0).sum())}/10")


## 1. The z-score window

The window sets both the rolling hedge-ratio lookback and the rolling mean/std of the
z-score. It was frozen at **60** on the reasoning that a trading quarter is a conventional
lookback. Nothing else about 60 is special, and any of 40–120 would have been equally easy
to justify in advance.


In [ ]:
WINDOWS = [40, 50, 60, 75, 90, 120]
window_sweep = pd.DataFrame(
    {w: oos(with_signal(cfg, window=w)) for w in WINDOWS}
).reindex(PAIRS)
window_sweep.columns.name = "window"
display(window_sweep.round(1))

window_summary = pd.DataFrame({
    "mean %": window_sweep.mean(),
    "median %": window_sweep.median(),
    "pairs profitable": (window_sweep > 0).sum(),
})
display(window_summary.T.round(2))


In [ ]:
swing = (window_sweep.max(axis=1) - window_sweep.min(axis=1)).sort_values(ascending=False)
flips = ((window_sweep > 0).any(axis=1) & (window_sweep <= 0).any(axis=1))

print(f"pairs whose sign changes somewhere in the grid: {int(flips.sum())} of {len(PAIRS)}")
print(f"pairs that never change sign: {sorted(window_sweep.index[~flips])}\n")
print("swing (max - min) across the grid, percentage points:")
print(swing.round(1).to_string())


**Nine of ten pairs change sign somewhere in this grid.** Only SPY/VOO is stable,
and it is stable because it loses money at every window — its edge is too small to be moved
by anything.

The swings are not marginal. UPS/FDX spans 68 percentage points and UNP/CSX 66, on a
parameter with no principled value.

**The frozen window is the most favourable one in the grid.** At 60 the cross-sectional mean
is the highest of the six; every other window gives a lower one, and most give a negative
one. This belongs in the headline rather than a footnote: **the study's least-unfavourable
result comes from the single window that was fixed in advance.** A genuinely pre-registered
study that had fixed 90 instead — with identical a priori justification — would be reporting
a clearly negative mean.

That is not an argument for switching to 90. It is the sharpest available demonstration of
the thesis: the reported answer is dominated by an arbitrary choice made before seeing any
data, which is another way of saying the study has very little power to measure what it set
out to measure.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name in PAIRS:
    ax.plot(WINDOWS, window_sweep.loc[name], marker="o", ms=3, lw=1, alpha=0.65, label=name)
ax.plot(WINDOWS, window_sweep.mean(), color="black", lw=2.5, marker="s", label="mean")
ax.axhline(0, color="0.5", lw=1)
ax.axvline(cfg.signal.window, color="firebrick", ls="--", lw=1)
ax.annotate(" frozen", (cfg.signal.window, ax.get_ylim()[1]), va="top",
            color="firebrick", fontsize=9)
ax.set_xlabel("z-score window (trading days)")
ax.set_ylabel("net out-of-sample total return %")
ax.set_title("Every pair's result moves with the window")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()


## 2. Entry and exit bands

Enter at |z| ≥ 2.0, exit at |z| ≤ 0.5. These are the textbook values and were frozen for
that reason. Entry controls how extreme a deviation must be before betting on reversion;
exit controls how much of the reversion to wait for. Wider entry means fewer, higher-
conviction trades; a lower exit means holding longer for a fuller reversion.

Sweeping both, and discarding combinations where exit ≥ entry (which would make the
hysteresis rule incoherent — `config.py` rejects them outright):


In [ ]:
ENTRIES, EXITS = [1.5, 2.0, 2.5, 3.0], [0.0, 0.25, 0.5, 1.0]

band_mean = pd.DataFrame(index=ENTRIES, columns=EXITS, dtype=float)
band_pos = pd.DataFrame(index=ENTRIES, columns=EXITS, dtype=float)
for e in ENTRIES:
    for x in EXITS:
        if x >= e:
            continue
        r = oos(with_signal(cfg, entry=e, exit=x))
        band_mean.loc[e, x] = r.mean()
        band_pos.loc[e, x] = (r > 0).sum()

band_mean.index.name = band_pos.index.name = "entry"
band_mean.columns.name = band_pos.columns.name = "exit"
print("mean net OOS total return %, across the 10 pairs:")
display(band_mean.round(2))
print("pairs profitable:")
display(band_pos)


In [ ]:
vals = band_mean.stack()
print(f"frozen setting (entry {cfg.signal.entry}, exit {cfg.signal.exit}): "
      f"{band_mean.loc[cfg.signal.entry, cfg.signal.exit]:+.2f}%")
print(f"best  in grid: {vals.max():+.2f}%  at entry/exit {vals.idxmax()}")
print(f"worst in grid: {vals.min():+.2f}%  at entry/exit {vals.idxmin()}")
print(f"\nall {len(vals)} combinations lie between {vals.min():+.2f}% and {vals.max():+.2f}% "
      f"— a spread of {vals.max() - vals.min():.2f}pp on a mean of {vals.mean():+.2f}%")


The bands matter **less** than the window does, which is mildly reassuring and
mostly uninformative. Every combination lands within a few percentage points of zero, and
the number of profitable pairs moves only between 4 and 7 out of 10.

There is no monotone structure — no "wider entry is better", no "hold longer is better".
The surface is flat and noisy, which is what a surface looks like when the underlying effect
is close to zero and the variation is sampling noise. The frozen 2.0/0.5 is unremarkable
within it, neither flattering nor unflattering.

This is the one sweep in the notebook where the frozen choice does **not** turn out to be
load-bearing.


## 3. Cost sensitivity, and the breakeven level

The study charges 1 bp commission + 5 bps slippage = **6 bps per side**, applied to
`(1 + |g|)` of notional on every unit of position change. That figure is a judgement call
about a retail taker in liquid US large caps, and it is the assumption a sceptical reader is
most likely to push back on.

So rather than defend it, sweep it — and find the level at which each pair breaks even.


In [ ]:
def with_cost(bps):
    return dataclasses.replace(
        cfg, costs=dataclasses.replace(cfg.costs, commission_bps=0.0, slippage_bps=bps)
    )


COSTS = [0.0, 1.0, 2.0, 3.0, 6.0, 10.0, 15.0, 20.0]
cost_sweep = pd.DataFrame({c: oos(with_cost(c)) for c in COSTS}).reindex(PAIRS)
cost_sweep.columns.name = "bps per side"
display(cost_sweep.round(1))

display(pd.DataFrame({
    "mean %": cost_sweep.mean(),
    "pairs profitable": (cost_sweep > 0).sum(),
}).T.round(2))


In [ ]:
# Breakeven: the lowest cost level at which a pair's net OOS return turns non-positive.
# Two cases need separating from a naive scan -- pairs already losing at zero cost, and
# pairs still profitable at the top of the grid.
GRID = np.arange(0, 201, 1.0)

breakeven = {}
for p in cfg.pairs:
    curve = np.array([
        run_pair(p, prices, with_cost(float(b))).metrics["out_of_sample"]["net"]["total_return"]
        for b in GRID
    ])
    if curve[0] <= 0:
        breakeven[p.name] = -1.0          # unprofitable even with zero costs
    elif curve[-1] > 0:
        breakeven[p.name] = np.inf        # still profitable at 200 bps
    else:
        breakeven[p.name] = float(GRID[np.argmax(curve <= 0)])

be = pd.Series(breakeven).sort_values(ascending=False)


def label(x):
    if x == -1:
        return "no edge at any cost"
    return ">200" if np.isinf(x) else f"{x:.0f}"


print(f"breakeven cost (bps/side), against the {cfg.costs.commission_bps + cfg.costs.slippage_bps:.0f} "
      f"bps actually charged:\n")
for name, v in be.items():
    print(f"  {name:<10} {label(v):>20}")

no_edge = int((be == -1).sum())
print(f"\nno gross edge at all: {no_edge}/10")
print(f"breakeven above the 6 bps charged: {int((be > 6).sum())}/10")
print(f"marginal (breakeven at or below 6 bps): {int(((be > 0) & (be <= 6)).sum())}/10")


**This is the most useful table in the notebook, and it reframes the cost story that
`03_backtest_results.ipynb` §3 tells.**

The pairs are **bimodal**, not spread along a spectrum:

- **Four pairs have no gross edge at any cost level** (WM/RSG, SPY/VOO, XOM/CVX, UPS/FDX).
  They lose money with commissions and slippage set to *zero*. No cost assumption can rescue
  them, and attributing their losses to transaction costs would be simply wrong.
- **Four pairs clear the 6 bps charge with enormous margin** — UNP/CSX breaks even at
  ~100 bps per side, DUK/SO at ~52, KO/PEP at ~44, HD/LOW at ~26. These are 4x to 16x the
  assumed cost. Whether the cost assumption is 3 bps or 10 bps changes nothing for them.
- **Only two pairs sit in the zone where the assumption matters at all** — MA/V (~2 bps) and
  FOXA/FOX (~1 bp), both of which are near zero either way.

So the honest version of the cost claim is narrower than "costs kill the edge". Costs
consume ~80% of the *mean* gross return, which is a real and large effect on the average,
but **for 8 of 10 individual pairs the cost level is nearly irrelevant to the sign.** The
strategy's failures are mostly failures of edge, not of friction.

The cross-sectional mean does cross zero between 6 and 10 bps per side, so the *aggregate*
verdict is genuinely cost-sensitive around the assumed level. That is worth knowing, and it
means a reader who thinks 6 bps is too harsh has a legitimate quarrel with the mean — but
not with the eight pairs whose verdicts do not move.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name in PAIRS:
    ax.plot(COSTS, cost_sweep.loc[name], marker="o", ms=3, lw=1, alpha=0.6, label=name)
ax.plot(COSTS, cost_sweep.mean(), color="black", lw=2.5, marker="s", label="mean")
ax.axhline(0, color="0.5", lw=1)
ax.axvline(6, color="firebrick", ls="--", lw=1)
ax.annotate(" charged", (6, ax.get_ylim()[1]), va="top", color="firebrick", fontsize=9)
ax.set_xlabel("cost, bps per side")
ax.set_ylabel("net out-of-sample total return %")
ax.set_title("Cost sensitivity: most pairs' verdicts do not depend on the assumption")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()


## 4. The split date

2021-12-31 was chosen as a round date leaving roughly 7 years in and 3 out. It was fixed
once and the out-of-sample period was scored once. But it is still a choice, and a different
one changes both how much data the hedge ratio is fitted on and which regime gets scored.

Comparing across split dates requires **annualized** returns — a longer out-of-sample window
mechanically accumulates more total return, so raw totals would not be comparable.


In [ ]:
SPLITS = ["2019-12-31", "2020-12-31", "2021-12-31", "2022-12-31"]

rows = []
for d in SPLITS:
    variant = dataclasses.replace(cfg, split=dataclasses.replace(cfg.split, in_sample_end=d))
    ann = oos(variant, field="annualized_return")
    rows.append({
        "split": d,
        "OOS years": round((pd.Timestamp(cfg.data.end) - pd.Timestamp(d)).days / 365.25, 1),
        "mean %/yr": ann.mean(),
        "median %/yr": ann.median(),
        "pairs profitable": int((ann > 0).sum()),
    })
pd.DataFrame(rows).set_index("split").round(2)


The mean annualized return wanders between roughly −0.5%/yr and +2.4%/yr with no
trend, and the count of profitable pairs stays at 4–5 of 10 throughout.

**Here the frozen choice is the *least* flattering of the four.** 2021-12-31 gives the
lowest mean of the grid. That cuts in the opposite direction to the window result in §1, and
it is worth saying plainly: the two most consequential frozen parameters happen to pull
opposite ways, so there is no case that the configuration was tilted toward a particular
answer. It was simply fixed, and the consequences fell where they fell.

The broader reading is the same as everywhere else in this notebook — a conclusion that
moves by 3 percentage points a year depending on which December you split on is not a
conclusion with much in it.


## 5. Hedge specification: rolling versus static signal

`02_cointegration_analysis.ipynb` §4 found that the rolling spread construction is
stationary for **10 of 10** pairs in-sample against **4 of 10** for the static one, which is
the stated ADF justification for `signal_hedge: rolling`. That looks like a decisive
statistical argument.

It is worth checking whether it is worth anything in returns. Note that the static branch
reuses the **in-sample-only** hedge ratio, so this is a fair comparison of two clean
estimators rather than a clean one against a leaking one.


In [ ]:
hedge = pd.DataFrame(
    {h: oos(with_signal(cfg, signal_hedge=h)) for h in ["rolling", "static"]}
).reindex(PAIRS)
hedge["difference"] = hedge["rolling"] - hedge["static"]
display(hedge.round(1))

display(pd.DataFrame({
    "mean %": hedge[["rolling", "static"]].mean(),
    "median %": hedge[["rolling", "static"]].median(),
    "pairs profitable": (hedge[["rolling", "static"]] > 0).sum(),
}).T.round(2))


**A 10-of-10 versus 4-of-10 advantage in stationarity buys essentially nothing on
average — while scrambling every individual pair.**

Mean net out-of-sample return is +0.64% with rolling and −0.38% with static: one point
apart, on a cross-section whose standard deviation is 26 points. The static specification
actually has the *better* median and one more profitable pair.

But look at the `difference` column rather than the summary. UNP/CSX is **+47.7 points**
better under rolling; MA/V is **30.2 points worse**; KO/PEP +25.6, XOM/CVX −22.9. The
specification does not nudge results, it reshuffles them — and the reshuffling cancels
almost exactly in the mean.

That pattern is the signature of noise, not of a better estimator. A specification with a
genuine edge would move pairs in a consistent direction; one that moves them ±48 points at
random and nets to nothing is picking up which side of a coin each pair happened to land on.
The study's single best performer under the configured specification (UNP/CSX, +55.9%) drops
to +8.2% under the other one — the headline pair is an artifact of a choice made for
unrelated statistical reasons.

This is a useful corrective, and it generalizes past this study. The ADF result in notebook
02 is real, but it largely measures the estimator rather than the pairs: a hedge ratio
re-fitted every 60 days produces a series that looks stationary almost by construction,
because it continuously refits the level it then measures deviations from. Passing that test
does not imply anything tradeable, and here it demonstrably does not.

The config keeps `rolling` because the a priori argument for it was made before any of these
returns existed, and switching now on the basis of this table would be exactly the
parameter-fitting the study refuses to do everywhere else.


## 6. Summary: what survives

Collecting the range of the cross-sectional mean under each single-parameter perturbation,
against the baseline of **+0.64%**:


In [ ]:
summary = pd.DataFrame([
    {"parameter": "z-score window", "varied over": "40-120 days",
     "min mean %": window_sweep.mean().min(), "max mean %": window_sweep.mean().max()},
    {"parameter": "entry / exit bands", "varied over": "entry 1.5-3.0, exit 0.0-1.0",
     "min mean %": band_mean.stack().min(), "max mean %": band_mean.stack().max()},
    {"parameter": "cost", "varied over": "0-20 bps/side",
     "min mean %": cost_sweep.mean().min(), "max mean %": cost_sweep.mean().max()},
    {"parameter": "signal hedge", "varied over": "rolling / static",
     "min mean %": hedge[["rolling", "static"]].mean().min(),
     "max mean %": hedge[["rolling", "static"]].mean().max()},
]).set_index("parameter")
summary["range pp"] = summary["max mean %"] - summary["min mean %"]
summary["baseline mean %"] = BASELINE.mean()
summary["baseline sd pp"] = BASELINE.std()
display(summary.round(2))

# Persisted for 05_writeup.ipynb to read, so the writeup quotes these numbers rather than
# transcribing them. Same discipline as metrics.csv: one computation, one source.
out = ROOT / cfg.output.results_dir / "sensitivity.csv"
out.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(out)
print(f"\nbaseline mean: {BASELINE.mean():+.2f}%")
print(f"cross-sectional sd of the baseline itself: {BASELINE.std():.1f}pp")
print(f"wrote {out.relative_to(ROOT)}")


Two conclusions, and they point the same way.

**1. No single parameter choice rescues the strategy.** Nothing in any sweep produces a
cross-sectional mean far from zero. The best cell anywhere in this notebook is a couple of
percentage points of total return over three years, which is not an edge. There is no
configuration hiding in here that a more diligent analyst would have found.

**2. The perturbation ranges are small compared with the dispersion.** Moving a parameter
shifts the mean by a few percentage points; the spread *across pairs* at any fixed setting
is 26 points. **Which pairs you picked matters far more than how you set any dial.** That is
the same conclusion `03_backtest_results.ipynb` §5 reaches from the cross-section and
`02_cointegration_analysis.ipynb` §2 reaches from the instability of cointegration itself —
three independent routes to it.

The uncomfortable corollary, and the reason this notebook exists: **a study that had frozen
its window at 90 instead of 60 would have reported a clearly negative headline from the same
data, the same code, and the same pairs.** Pre-registration is what makes a result
interpretable, but it does not make it *stable* — and reporting the sweep is the only way a
reader can tell the difference.

**Next:** `05_writeup.ipynb` assembles this with the rest of the evidence.
